# Day 026 Project Solution — AutoReporter

An `AutoReporter` that turns raw data dicts into styled PDF and DOCX reports using AI-generated section content.

In [ ]:
import json, os, tempfile
from pypdf import PdfReader
from docx import Document
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from xml.sax.saxutils import escape
import ollama


def read_pdf_pages(pdf_path: str) -> list[str]:
    reader = PdfReader(pdf_path)
    return [page.extract_text() or "" for page in reader.pages]


def read_docx_text(docx_path: str) -> str:
    doc = Document(docx_path)
    return "\n".join(p.text for p in doc.paragraphs)


def create_pdf_report(title: str, sections: list[dict], output_path: str) -> None:
    doc = SimpleDocTemplate(output_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [
        Paragraph(escape(title), styles["Title"]),
        Spacer(1, 0.25 * inch),
    ]
    for section in sections:
        story.append(Paragraph(escape(section["heading"]), styles["Heading1"]))
        story.append(Paragraph(escape(section["body"]), styles["Normal"]))
        story.append(Spacer(1, 0.15 * inch))
    doc.build(story)


def create_docx_report(title: str, sections: list[dict], output_path: str) -> None:
    doc = Document()
    doc.add_heading(title, level=0)
    for section in sections:
        doc.add_heading(section["heading"], level=1)
        doc.add_paragraph(section["body"])
    doc.save(output_path)


def ai_generate_section(
    topic: str,
    data_snippet: str,
    model: str = "llama3.2",
) -> dict:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional report writer. "
                    "Generate a concise report section from the data provided. "
                    'Return JSON with exactly two keys: "heading" (a short title string) '
                    'and "body" (2-4 sentences of professional analysis). '
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Topic: {topic}\n\n"
                    f"Data:\n{data_snippet[:500]}\n\n"
                    "Generate a report section:"
                ),
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        result = json.loads(raw)
        return {
            "heading": str(result.get("heading", topic)),
            "body": str(result.get("body", "")),
        }
    except Exception:
        return {"heading": topic, "body": raw}


class AutoReporter:
    def __init__(self, model: str = "llama3.2"):
        self.model = model

    def generate_sections(self, data: dict) -> list[dict]:
        sections = []
        for key, value in data.items():
            snippet = (
                json.dumps(value, indent=2)
                if isinstance(value, (dict, list))
                else str(value)
            )
            section = ai_generate_section(key, snippet, model=self.model)
            sections.append(section)
        return sections

    def to_pdf(self, title: str, sections: list[dict], output_path: str) -> None:
        create_pdf_report(title, sections, output_path)

    def to_docx(self, title: str, sections: list[dict], output_path: str) -> None:
        create_docx_report(title, sections, output_path)

    def generate_report(self, data: dict, title: str, output_dir: str = ".") -> dict:
        import os
        sections = self.generate_sections(data)
        pdf_path = os.path.join(output_dir, "report.pdf")
        docx_path = os.path.join(output_dir, "report.docx")
        self.to_pdf(title, sections, pdf_path)
        self.to_docx(title, sections, docx_path)
        return {"pdf": pdf_path, "docx": docx_path, "sections": sections}

In [ ]:
SAMPLE_DATA = {
    'Sales Performance': 'Q1 revenue: 1.2M, up 15 percent YoY. Top product: Widget A with 38 percent share.',
    'Market Overview':   'Total market: 50B. Our share: 2.4 percent. Three main competitors.',
}

## Action 1 — Generate Report Sections

In [ ]:
reporter = AutoReporter()
print('Generating report sections (2 AI calls)...')
sections = reporter.generate_sections(SAMPLE_DATA)
print(f'Generated {len(sections)} section(s):')
for s in sections:
    print(f"  - {s['heading']}")

## Action 2 — Write PDF and DOCX

In [ ]:
tmp_dir = tempfile.gettempdir()
pdf_path  = os.path.join(tmp_dir, 'day026_report.pdf')
docx_path = os.path.join(tmp_dir, 'day026_report.docx')

reporter.to_pdf('Q1 Business Report', sections, pdf_path)
reporter.to_docx('Q1 Business Report', sections, docx_path)
print(f'PDF:  {pdf_path} ({os.path.getsize(pdf_path):,} bytes)')
print(f'DOCX: {docx_path} ({os.path.getsize(docx_path):,} bytes)')

## Action 3 — Read Back and Verify

In [ ]:
pages = read_pdf_pages(pdf_path)
print(f'PDF: {len(pages)} page(s), first 120 chars:')
print((pages[0] if pages else '(no text)').strip()[:120])

body = read_docx_text(docx_path)
print(f'\nDOCX: {len(body)} chars, preview:')
print(body[:120])
print('\nReport generation complete!')